<h2> Imports & Configuration </h2>

> 🧭 **Modern Data Engineering Ecosystem Navigation**
> - 📘 **Architecture & Concept:** [Data System Design — Storage Optimization & Clustering](https://github.com/santoshjammi29/data-system-design-handbook/tree/main/modules/04_storage_optimization_and_clustering)
> - 💻 **Hands-On Recipe:** [PySpark Handbook — Notebook 08_03: Data Skew Mitigation](https://github.com/santoshjammi29/pyspark-handbook/blob/main/notebooks/08_performance_and_tuning/08_03_data_skew_mitigation_salting.py)
> - 🔬 **Empirical Lab:** [Spark Experiments — 1_data_skew: Salting Benchmarks](https://github.com/santoshjammi29/spark-experiments/blob/main/spark/1_data_skew/4_salting.ipynb) *(You are here)*
> - 🚨 **Incident Runbook:** [PySpark Runbook 03: Data Skew & Two-Phase Salting](https://github.com/santoshjammi29/pyspark-handbook/blob/main/runbooks/03_data_skew_and_two_phase_salting.md)
> - 🎯 **Interview Defense:** [Interview Handbook — Pillar 01: Spark & Databricks Q12](https://github.com/santoshjammi29/data-engineering-interview-handbook/blob/main/pillars/01_spark_and_databricks.md#q12)
> - 🌐 **Interactive Web Practice:** [Practice on moderndataengineer.uk](https://moderndataengineer.uk/practice/pyspark?topic=data-skew)
> - 📐 **Architecture Blueprint:** [Data Skew Mitigation & Key Salting Architecture](https://moderndataengineer.uk/whiteboards?id=data_skew_salting_repartition)


In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [ ]:
from pyspark.sql.types import IntegerType
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "3")
spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.adaptive.enabled", "false")

<h2> Simulating Skewed Join </h2>

In [ ]:
df_uniform = spark.createDataFrame([i for i in range(1000000)], IntegerType())
df_uniform.show(5, False)

In [ ]:
(
    df_uniform
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .count()
    .orderBy("partition")
    .show(15, False)
)

In [ ]:
df0 = spark.createDataFrame([0] * 999990, IntegerType()).repartition(1)
df1 = spark.createDataFrame([1] * 15, IntegerType()).repartition(1)
df2 = spark.createDataFrame([2] * 10, IntegerType()).repartition(1)
df3 = spark.createDataFrame([3] * 5, IntegerType()).repartition(1)
df_skew = df0.union(df1).union(df2).union(df3)
df_skew.show(5, False)

In [ ]:
(
    df_skew
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .count()
    .orderBy("partition")
    .show()
)

In [ ]:
df_joined_c1 = df_skew.join(df_uniform, "value", 'inner')

In [ ]:
(
    df_joined_c1
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .count()
    .show(5, False)
)

<h2> Simulating Uniform Distribution Through Salting </h2>

In [ ]:
SALT_NUMBER = int(spark.conf.get("spark.sql.shuffle.partitions"))
SALT_NUMBER

In [ ]:
df_skew = df_skew.withColumn("salt", (F.rand() * SALT_NUMBER).cast("int"))

In [ ]:
df_skew.show(10, truncate=False)

In [ ]:
df_uniform = (
    df_uniform
    .withColumn("salt_values", F.array([F.lit(i) for i in range(SALT_NUMBER)]))
    .withColumn("salt", F.explode(F.col("salt_values")))
)

In [ ]:
df_uniform.show(10, truncate=False)

In [ ]:
df_joined = df_skew.join(df_uniform, ["value", "salt"], 'inner')

In [ ]:
(
    df_joined
    .withColumn("partition", F.spark_partition_id())
    .groupBy("value", "partition")
    .count()
    .orderBy("value", "partition")
    .show()
)

### 📊 Visual Evidence: Partition Rebalance via Two-Phase Salting
![Before and After Salting Partition Rebalance](../../docs/images/generated/salting_partition_rebalance.png)

| Partition | Unsalted Rows | Salted Rows (3-Way Salt) | Rebalance Delta |
| :--- | :--- | :--- | :--- |
| **Partition 0** | **999,990 (99.999%)** | **332,774 (33.28%)** | **-66.7% (Resolved skew)** |
| **Partition 1** | 15 (0.001%) | **333,601 (33.36%)** | **+33.36% (Uniform share)** |
| **Partition 2** | 10 (0.001%) | **333,615 (33.36%)** | **+33.36% (Uniform share)** |
| **Partition 3** | 5 (0.001%) | N/A (3 partitions configured) | N/A |

#### 🔬 Mechanics of Two-Phase Key Salting:
1. **The Problem**: A single hot key (`value = 0`) concentrates 999,990 rows into Partition 0, starving the other executor cores.
2. **Phase 1 (Salt Generation)**: The skewed table adds a random salt suffix `[0 .. N-1]` where `N = 3`:
   `df_skew = df_skew.withColumn("salt", (F.rand() * 3).cast("int"))`
3. **Phase 1b (Replication/Explosion)**: The uniform/dimension table is exploded with all possible salt values `[0 .. N-1]` so every lookup key can still match.
4. **Phase 2 (Salted Join)**: Joining on `["value", "salt"]` redistributes the 1,000,000 rows across 3 partitions with ~333,333 rows each, achieving uniform core utilization.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../../docs/images/generated/salting_partition_rebalance.png"))


# Salting In Aggregations

In [ ]:
df_skew.groupBy("value").count().show()

In [ ]:
(
    df_skew
    .withColumn("salt", (F.rand() * SALT_NUMBER).cast("int"))
    .groupBy("value", "salt")
    .agg(F.count("value").alias("count"))
    .groupBy("value")
    .agg(F.sum("count").alias("count"))
    .show()
)

In [ ]:
spark.stop()